<a href="https://colab.research.google.com/github/yshzjq/Step-2.Statistical_Thinking-Machine-Learning/blob/main/Step%202.%20%ED%86%B5%EA%B3%84%EC%A0%81%20%EC%82%AC%EA%B3%A0%EC%99%80%20%EB%A8%B8%EC%8B%A0%EB%9F%AC%EB%8B%9D%20%EC%9E%85%EB%AC%B8/1_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 데이터 처리용 라이브러리
import pandas as pd
import numpy as np

# 스케일링 계산
from sklearn.preprocessing import StandardScaler


# 시각화 라이브러리
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# 데이터 URL
path ="/content/Shopping_Mall_Customer_Segmentation_Data_.csv"
# 로컬에서 여는 경우 경로 수정 필요!

# CSV 파일 읽기
df = pd.read_csv(path)

# 데이터 확인
df.head()



,Customer ID,Age,Gender,Annual Income,Spending Score
0,d410ea53-6661-42a9-ad3a-f554b05fd2a7,30,Male,151479,89
1,1770b26f-493f-46b6-837f-4237fb5a314e,58,Female,185088,95
2,e81aa8eb-1767-4b77-87ce-1620dc732c5e,62,Female,70912,76
3,9795712a-ad19-47bf-8886-4f997d6046e3,23,Male,55460,57
4,64139426-2226-4cd6-bf09-91bce4b4db5e,24,Male,153752,76


In [5]:
# 고객 특징으로 사용할 컬럼 선택
features = ['Age','Annual Income','Spending Score']

# 특징 데이터만 추출
X = df[features]

# 확인
X.head()



,Age,Annual Income,Spending Score
0,30,151479,89
1,58,185088,95
2,62,70912,76
3,23,55460,57
4,24,153752,76


In [6]:
# 스케일링
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# 기준 고객(원하는 번호로 바꾸면 됨)
target_idx = 0
target = X_scaled.iloc[target_idx]

target

,0
Age,-1.145516
Annual Income,0.798813
Spending Score,1.337059


In [7]:
# 유사도/거리 계산 함수 3개

def euclidean(a, b):
    # 유클리드 거리: 직선 거리 (작을수록 유사)
    return np.sqrt(np.sum((a - b) ** 2))

def manhattan(a, b):
    # 맨해튼 거리: 차이의 합 (작을수록 유사)
    return np.sum(np.abs(a - b))

def cosine_sim(a, b):
    # 코사인 유사도: 방향(패턴) 유사 (클수록 유사)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

In [9]:
def find_topk(metric_name, metric_func, bigger_is_better, k):
    scores = []  # (인덱스, 점수) 담을 리스트

    for i in range(len(X_scaled)):
        if i == target_idx:
            continue  # 자기 자신은 제외

        score = metric_func(target, X_scaled.iloc[i])  # 점수 계산
        scores.append((i, score))  # 저장

    # 정렬 방향 결정
    scores.sort(key=lambda x: x[1], reverse=bigger_is_better)

    # Topk 선택
    topk = scores[:k]

    # 보기 좋은 표로 만들기
    result = df.loc[[idx for idx, _ in topk], features].copy()
    result[metric_name] = [score for _, score in topk]

    return result

In [10]:
# 유사도 3종 실행
print("유클리드(작을수록 유사)")
display(find_topk("euclidean_dist", euclidean, False, 3))

print("맨해튼(작을수록 유사)")
display(find_topk("manhattan_dist", manhattan, False, 3))

print("코사인(클수록 유사)")
display(find_topk("cosine_sim", cosine_sim, True, 3))

유클리드(작을수록 유사)


,Age,Annual Income,Spending Score,euclidean_dist
2953,31,150792,91,0.085220
4628,32,152904,90,0.104520
8404,28,154078,87,0.127635


맨해튼(작을수록 유사)


,Age,Annual Income,Spending Score,manhattan_dist
2953,31,150792,91,0.130124
4628,32,152904,90,0.156789
4443,29,145488,90,0.196829


코사인(클수록 유사)


,Age,Annual Income,Spending Score,cosine_sim
7079,25,162754,97,0.999813
4065,26,155760,96,0.999703
8826,29,153986,93,0.999636
